# Module 3 – Logic Gates with a Perceptron
**Course:** ECE 5831 – Pattern Recognition and Neural Networks (Fall 2026)  
**Assignment:** A_03

This notebook documents and tests the `LogicGate` class in `logic_gate.py`, which implements AND, NAND, OR, NOR and XOR gates using a perceptron and NumPy.

## 1. Background: the perceptron
A two-input perceptron outputs 1 when the weighted sum of its inputs exceeds a threshold $\theta$:

$$y = \begin{cases}1 & w_1x_1 + w_2x_2 > \theta\\ 0 & \text{otherwise}\end{cases}$$

A larger weight means that input matters more. With NumPy the weighted sum is `np.dot(x, w)` (element-wise multiply, then sum).

## 2. Import the class
The class lives in `logic_gate.py`. Each gate stores its weights, threshold, inputs and last output on the object.

In [1]:
import numpy as np
import logic_gate as lg

gate = lg.LogicGate()
inputs = [(0, 0), (0, 1), (1, 0), (1, 1)]

## 3. The core computation with NumPy
Every gate does exactly this with different weights and threshold (shown here for AND: $w=(0.5,0.5)$, $\theta=0.7$).

In [2]:
x = np.array([1, 1])
w = np.array([0.5, 0.5])
th = 0.7
print("x*w          =", x * w)
print("np.dot(x, w) =", np.dot(x, w))
print("output       =", 1 if np.dot(x, w) > th else 0)

x*w          = [0.5 0.5]
np.dot(x, w) = 1.0
output       = 1


## 4. Helper to show a gate's truth table
Runs a gate on all four input combinations and uses the class's `print_output` method.

In [3]:
def show(name):
    func = getattr(gate, name.lower() + "_gate")
    for x1, x2 in inputs:
        func(x1, x2)
        gate.print_output(name)

## 5. AND gate
Weights $0.5, 0.5$ and threshold $0.7$. Only (1,1) gives $1.0 > 0.7$.

In [4]:
show('AND')

Output of AND logic is: 0, with x1 = 0, x2 = 0
Output of AND logic is: 0, with x1 = 0, x2 = 1
Output of AND logic is: 0, with x1 = 1, x2 = 0
Output of AND logic is: 1, with x1 = 1, x2 = 1


## 6. NAND gate
Negative weights $-1,-1$ and threshold $-1.5$: the inverse of AND. NAND is a *universal* gate.

In [5]:
show('NAND')

Output of NAND logic is: 1, with x1 = 0, x2 = 0
Output of NAND logic is: 1, with x1 = 0, x2 = 1
Output of NAND logic is: 1, with x1 = 1, x2 = 0
Output of NAND logic is: 0, with x1 = 1, x2 = 1


## 7. OR gate
Weights $0.5, 0.5$ and threshold $0.0$: any single 1 input gives a sum above 0.

In [6]:
show('OR')

Output of OR logic is: 0, with x1 = 0, x2 = 0
Output of OR logic is: 1, with x1 = 0, x2 = 1
Output of OR logic is: 1, with x1 = 1, x2 = 0
Output of OR logic is: 1, with x1 = 1, x2 = 1


## 8. NOR gate
Negative weights $-1,-1$ and threshold $-0.5$: the inverse of OR. Only (0,0) gives $0 > -0.5$.

In [7]:
show('NOR')

Output of NOR logic is: 1, with x1 = 0, x2 = 0
Output of NOR logic is: 0, with x1 = 0, x2 = 1
Output of NOR logic is: 0, with x1 = 1, x2 = 0
Output of NOR logic is: 0, with x1 = 1, x2 = 1


## 9. XOR gate – the limitation of a single perceptron
XOR is **not linearly separable**: no single choice of $w_1, w_2, \theta$ works. The fix is a **multilayer perceptron**:

$$\text{XOR}(x_1,x_2)=\text{AND}\big(\text{OR}(x_1,x_2),\ \text{NAND}(x_1,x_2)\big)$$

Layer 1 computes OR and NAND; layer 2 ANDs the results.

In [8]:
show('XOR')

Output of XOR logic is: 0, with x1 = 0, x2 = 0
Output of XOR logic is: 1, with x1 = 0, x2 = 1
Output of XOR logic is: 1, with x1 = 1, x2 = 0
Output of XOR logic is: 0, with x1 = 1, x2 = 1


## 10. Verification against expected truth tables
Assert that every gate matches the known correct outputs.

In [9]:
expected = {
    'AND':  [0, 0, 0, 1],
    'NAND': [1, 1, 1, 0],
    'OR':   [0, 1, 1, 1],
    'NOR':  [1, 0, 0, 0],
    'XOR':  [0, 1, 1, 0],
}
for name, exp in expected.items():
    func = getattr(gate, name.lower() + "_gate")
    out = [func(a, b) for a, b in inputs]
    assert out == exp, (name, out, exp)
    print(f"{name:5s} {out}  OK")

AND   [0, 0, 0, 1]  OK
NAND  [1, 1, 1, 0]  OK
OR    [0, 1, 1, 1]  OK
NOR   [1, 0, 0, 0]  OK
XOR   [0, 1, 1, 0]  OK


## 11. Summary table
All five gates side by side.

In [10]:
print("x1 x2 | AND NAND OR NOR XOR")
for a, b in inputs:
    r = [gate.and_gate(a,b), gate.nand_gate(a,b), gate.or_gate(a,b), gate.nor_gate(a,b), gate.xor_gate(a,b)]
    print(f" {a}  {b} |  {r[0]}    {r[1]}   {r[2]}   {r[3]}   {r[4]}")

x1 x2 | AND NAND OR NOR XOR
 0  0 |  0    1   0   1   0
 0  1 |  0    1   1   0   1
 1  0 |  0    1   1   0   1
 1  1 |  1    0   1   0   0


## 12. Conclusion
A single perceptron implements AND, NAND, OR and NOR by choosing weights and a threshold; XOR needs two layers. Finding good weights automatically is what neural network *training* does.